<a href="https://colab.research.google.com/github/Balajivallepu/DL/blob/main/DL_LAB_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
from datasets import load_dataset

dataset = load_dataset("wangrongsheng/ag_news")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [39]:
from datasets import load_dataset
import pandas as pd

In [40]:
dataset = load_dataset("wangrongsheng/ag_news")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [41]:
import re

def clean_text(text):
    text = text.lower()                  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation, numbers
    return text

train_text = [clean_text(x["text"]) for x in train_data]
test_text = [clean_text(x["text"]) for x in test_data]

train_labels = [x["label"] for x in train_data]
test_labels = [x["label"] for x in test_data]

print(train_text[0])

wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again


In [42]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(train_text)

train_sequences = tokenizer.texts_to_sequences(train_text)
test_sequences = tokenizer.texts_to_sequences(test_text)

print(train_sequences[0])

[392, 325, 1526, 1, 100, 55, 2, 813, 24, 24, 1, 392, 1989, 1, 5, 1, 35, 3894, 738, 296]


In [43]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 50

train_padded = pad_sequences(train_sequences,
                             maxlen=max_length,
                             padding='post',
                             truncating='post')

test_padded = pad_sequences(test_sequences,
                            maxlen=max_length,
                            padding='post',
                            truncating='post')

print(train_padded.shape)


(120000, 50)


In [44]:
from tensorflow.keras.utils import to_categorical

num_classes = 4

train_labels = to_categorical(train_labels,num_classes)
test_labels = to_categorical(test_labels,num_classes)

print(train_labels[0])

[0. 0. 1. 0.]


In [45]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

embedding_dim = 64

model = Sequential([
    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              input_length=max_length),

    SimpleRNN(64),

    Dense(num_classes,activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [46]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_padded,
    train_labels,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 42s 27ms/step - accuracy: 0.8304 - loss: 0.4945 - val_accuracy: 0.8646 - val_loss: 0.4057
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 26ms/step - accuracy: 0.8995 - loss: 0.3215 - val_accuracy: 0.8622 - val_loss: 0.3983
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 26ms/step - accuracy: 0.9165 - loss: 0.2672 - val_accuracy: 0.8718 - val_loss: 0.3872
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 26ms/step - accuracy: 0.9268 - loss: 0.2345 - val_accuracy: 0.8628 - val_loss: 0.4267
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 26ms/step - accuracy: 0.9322 - loss: 0.2151 - val_accuracy: 0.8718 - val_loss: 0.4026


In [47]:
loss, accuracy = model.evaluate(test_padded, test_labels)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8784 - loss: 0.3913
Test Loss : 0.39129069447517395
Test Accuracy : 0.8784210681915283


In [48]:
news = ["Apple launches a new AI-powered iPhone"]

news = [clean_text(x) for x in news]

sequence = tokenizer.texts_to_sequences(news)

padded = pad_sequences(sequence,maxlen=max_length,padding='post')

prediction = model.predict(padded)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
[[3.3776391e-02 2.4758215e-04 3.1933829e-03 9.6278268e-01]]


In [49]:
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

predicted_class = prediction.argmax()

print("Predicted Category :", labels[predicted_class])

Predicted Category : Sci/Tech
